In [0]:
import json
import re
from pyspark.sql import functions as F

# Validación de la ingesta de procesos SECOP II
Este notebook comprueba los procesos descargados por Data Factory para los contratos de **IDARTES** e **IDIPRON** de la cohorte 2024.
Reconstruye los lotes esperados a partir de los contratos, revisa los archivos y calcula la cobertura de la relación contrato-portafolio-proceso.
## 1. Parámetros de las ejecuciones completas
| Parámetro | Descripción |
| --- | --- |
| run_procesos | UUID del pipeline padre que agrupó los 70 lotes bajo la misma carpeta run= |
| run_idartes | UUID de la ingesta completa de contratos IDARTES |
| run_idipron | UUID de la ingesta completa de contratos IDIPRON |
| anio | Año de firma de los contratos; no filtra el año de publicación de los procesos |

Los valores predeterminados corresponden a la cohorte auditada. Si los cambias, usa UUID sin el prefijo run= y adapta los conteos esperados para otra cohorte.

In [0]:
dbutils.widgets.text("run_procesos", "6717455f-416d-4457-9a88-600732d976d6")
dbutils.widgets.text("run_idartes", "95aa87aa-90ee-4104-8f38-d18edc3287b8")
dbutils.widgets.text("run_idipron", "90a6911a-40da-486b-b7c4-9ac9ab37ecfa")
dbutils.widgets.text("anio", "2024")
dbutils.widgets.text("ambiente", "dev", "Ambiente: dev o prod")
dbutils.widgets.text("storage_account", "addb01", "Storage account del ambiente")

## 2. Validación de parámetros y acceso a Raw
Se valida el rango del año y el formato UUID de los tres Run IDs antes de leer archivos. Las rutas apuntan al contenedor `addb01/raw` mediante abfss. Los contratos se buscan en `secop/pruebas_paginacion/contratos/` y los procesos en `secop/procesos/`. 


In [0]:
ambiente = dbutils.widgets.get("ambiente").strip().lower()
storage_account = dbutils.widgets.get("storage_account").strip()

if ambiente not in {"dev", "prod"}:
    raise ValueError("ambiente debe ser dev o prod")

if not re.fullmatch(r"[a-z0-9]{3,24}", storage_account):
    raise ValueError("Nombre de storage account invalido")

if ambiente == "prod" and storage_account == "addb01":
    raise ValueError("Para prod debes indicar su storage account")

raiz = f"abfss://raw@{storage_account}.dfs.core.windows.net/"

print(f"Ambiente: {ambiente}")
print(f"Origen Raw: {raiz}")
anio = int(dbutils.widgets.get("anio"))
if not 2015 <= anio <= 2099:
    raise ValueError("Anio fuera de rango")

Ambiente: dev
Origen Raw: abfss://raw@addb01.dfs.core.windows.net/


In [0]:
def leer_uuid(nombre):
    valor = dbutils.widgets.get(nombre).strip()
    if not re.fullmatch(r"[0-9a-fA-F]{8}(?:-[0-9a-fA-F]{4}){3}-[0-9a-fA-F]{12}", valor):
        raise ValueError(f"UUID invalido: {nombre}")
    return valor

In [0]:
run_procesos = leer_uuid("run_procesos")
runs_contratos = {entidad: leer_uuid(f"run_{entidad}") for entidad in ("idartes", "idipron")}

## 3. Reglas para comprobar la paginación
Cada lote contiene hasta 100 identificadores de portafolio, pero una página contiene hasta 100 filas de procesos.
| Respuesta de la consulta | Páginas esperadas |
| --- | --- |
| 90 filas | Una página con 90 |
| 135 filas | Una con 100 y otra con 35 |
| Exactamente 100 filas | Una con 100 y otra vacía que confirma el fin |

Si no hay archivos para un lote, no se asume que su consulta devolvió cero: falta evidencia para distinguir un lote vacío de una descarga ausente.
Estos controles están alineados con el tamaño de página y límite configurados en ADF. Si se cambian allí, también deben adaptarse aquí.

In [0]:
def comprobar_paginas(conteos):
    """Paginas consecutivas de 100, con ultima pagina de 0 a 99 filas."""
    numeros = sorted(conteos)
    if not numeros or numeros != list(range(1, len(numeros) + 1)):
        raise ValueError(f"Paginas faltantes o numeracion incorrecta: {numeros}")
    if len(numeros) > 10:
        raise ValueError("Lote excede el limite configurado de 10 paginas")
    if any(conteos[n] != 100 for n in numeros[:-1]):
        raise ValueError(f"Pagina intermedia no completa: {conteos}")
    if not 0 <= conteos[numeros[-1]] < 100:
        raise ValueError(f"No hay pagina final menor que 100 registros: {conteos}")

## 4. Auditoría por entidad: contratos, lotes y procesos
- **Contratos:** Exige el conteo esperado y la unicidad de id_contrato. Comprueba identificadores no vacíos, NIT, año de firma y formato de proceso_de_compra.
- **Lotes esperados:** Extrae portafolios distintos, los ordena y los divide en grupos de 100, reproduciendo la lógica de `01_preparar_lotes_procesos`.
- **Carpetas y archivos:** Exige exactamente los lotes esperados y nombres lote. Rechaza elementos inesperados y lotes sin archivos.
- **Lectura:** multiLine reconoce arrays JSON. FAILFAST detiene la auditoría ante JSON malformado. Se exigen `id_del_proceso` e `id_del_portafolio`.
- **Trazabilidad:** _metadata.file_path permite asociar cada fila con su lote y página. Los identificadores de proceso y portafolio no pueden ser nulos o vacíos.
- **Pertenencia:** Un left_anti join detecta procesos cuyo portafolio no estaba solicitado en su lote. Un resultado fuera del lote detiene la validación.
- **Paginación:** Agrupa filas por archivo y aplica las reglas de la sección anterior. Los JSON vacíos listados se contabilizan con cero filas.
- **Duplicados:** Un hash del contenido JSON normalizado permite medir repeticiones de filas completas. No es una comparación byte a byte del archivo ni una deduplicación.
- **Cobertura:** Relaciona `contratos.proceso_de_compra` con `procesos.id_del_portafolio` mediante left_semi join, conservando el grano contrato sin multiplicarlo por varias filas de procesos.

Un mismo ID de proceso puede tener variantes distintas, por eso filas_procesos, ids_proceso_distintos y filas_repetidas_por_contenido son métricas diferentes. Un portafolio puede corresponder a varios contratos: contratos_con_proceso no tiene por qué coincidir con portafolios_encontrados.
Las cachés se liberan al terminar cada entidad, incluso ante un error.

In [0]:
def revisar_entidad(entidad, nit, esperado):
    ruta_contratos = (raiz + f"secop/pruebas_paginacion/contratos/{entidad}/"
                      f"anio={anio}/run={runs_contratos[entidad]}/pagina_*.json")
    contratos = (spark.read.option("multiLine", "true").option("mode", "FAILFAST")
                 .option("primitivesAsString", "true").json(ruta_contratos)
                 .select("id_contrato", "proceso_de_compra", "nit_entidad", "fecha_de_firma"))
    contratos = contratos.cache()
    procesos = None
    try:
        total_contratos = contratos.count()
        if total_contratos != esperado or contratos.select("id_contrato").distinct().count() != esperado:
            raise ValueError(f"{entidad}: conteo de contratos o unicidad incorrectos")
        invalidos = (F.col("id_contrato").isNull() | (F.trim("id_contrato") == "")
                     | F.col("nit_entidad").isNull() | (F.col("nit_entidad") != nit)
                     | F.col("fecha_de_firma").isNull() | ~F.col("fecha_de_firma").startswith(f"{anio}-")
                     | F.col("proceso_de_compra").isNull()
                     | ~F.col("proceso_de_compra").rlike(r"^CO1\.BDOS\.\d+$"))
        if contratos.filter(invalidos).limit(1).count():
            raise ValueError(f"{entidad}: contratos fuera de alcance o con identificadores invalidos")
        portafolios = sorted(r[0] for r in contratos.select("proceso_de_compra").distinct().collect())
        asignacion = [(p, i // 100 + 1) for i, p in enumerate(portafolios)]
        esperados = spark.createDataFrame(asignacion, "id_del_portafolio string, lote int")
        total_lotes = (len(portafolios) + 99) // 100
        ruta_procesos = raiz + f"secop/procesos/{entidad}/anio_contratos={anio}/run={run_procesos}/"
        directorios = dbutils.fs.ls(ruta_procesos)
        carpetas = {}
        for entrada in directorios:
            coincidencia = re.fullmatch(r"lote=([1-9]\d*)/", entrada.name)
            if not entrada.isDir() or not coincidencia:
                raise ValueError(f"Elemento inesperado en la ingesta: {entrada.path}")
            carpetas[int(coincidencia[1])] = entrada.path
        if set(carpetas) != set(range(1, total_lotes + 1)):
            raise ValueError(f"{entidad}: faltan o sobran carpetas. Esperados {total_lotes}; encontrados {sorted(carpetas)}")
        archivos = {}
        for lote, ruta in sorted(carpetas.items()):
            paginas = {}
            for archivo in dbutils.fs.ls(ruta):
                coincidencia = re.fullmatch(r"pagina_([1-9]\d*)\.json", archivo.name)
                if archivo.isDir() or not coincidencia:
                    raise ValueError(f"Archivo inesperado: {archivo.path}")
                paginas[int(coincidencia[1])] = archivo.path
            if not paginas:
                raise ValueError(f"{entidad}, lote {lote}: sin archivos. No se puede confirmar que sea un lote vacio")
            archivos[lote] = paginas
        rutas = [ruta for paginas in archivos.values() for ruta in paginas.values()]
        base = (spark.read.option("multiLine", "true").option("mode", "FAILFAST")
                .option("primitivesAsString", "true").json(rutas))
        requeridas = {"id_del_proceso", "id_del_portafolio"}
        if not requeridas.issubset(base.columns):
            raise ValueError(f"{entidad}: faltan columnas de procesos: {requeridas - set(base.columns)}")
        huella = F.sha2(F.to_json(F.struct(*[F.col(c) for c in sorted(base.columns)])), 256)
        procesos = (base.select("id_del_proceso", "id_del_portafolio", huella.alias("huella"),
                                F.col("_metadata.file_path").alias("ruta_archivo"))
                    .withColumn("lote", F.regexp_extract("ruta_archivo", r"/lote=(\d+)/", 1).cast("int"))
                    .withColumn("pagina", F.regexp_extract("ruta_archivo", r"/pagina_(\d+)\.json$", 1).cast("int"))
                    .cache())
        malos = (F.col("id_del_proceso").isNull() | (F.trim("id_del_proceso") == "")
                 | F.col("id_del_portafolio").isNull() | (F.trim("id_del_portafolio") == "")
                 | F.col("lote").isNull() | F.col("pagina").isNull())
        if procesos.filter(malos).limit(1).count():
            raise ValueError(f"{entidad}: proceso o archivo con identificador invalido")
        if procesos.join(esperados, ["id_del_portafolio", "lote"], "left_anti").limit(1).count():
            raise ValueError(f"{entidad}: un proceso no pertenece a los portafolios solicitados para su lote")
        observados = {(r.lote, r.pagina): r['count'] for r in procesos.groupBy("lote", "pagina").count().collect()}
        detalles = []
        for lote, paginas in sorted(archivos.items()):
            conteos = {pagina: observados.get((lote, pagina), 0) for pagina in paginas}
            try:
                comprobar_paginas(conteos)
            except ValueError as error:
                raise ValueError(f"{entidad}, lote {lote}: {error}") from error
            detalles.append((entidad, lote, len(paginas), sum(conteos.values()), conteos[max(conteos)]))
        filas = procesos.count()
        contenido_distinto = procesos.select("huella").distinct().count()
        procesos_distintos = procesos.select("id_del_proceso").distinct().count()
        portafolios_encontrados = procesos.select("id_del_portafolio").distinct()
        # Semijoin conserva el grano contrato; no multiplica contratos por procesos repetidos.
        con_proceso = contratos.join(portafolios_encontrados,
                                     contratos.proceso_de_compra == portafolios_encontrados.id_del_portafolio,
                                     "left_semi").count()
        multiples = procesos.groupBy("id_del_portafolio").agg(
            F.countDistinct("id_del_proceso").alias("procesos_distintos")
        ).filter("procesos_distintos > 1").count()
        resumen = {"entidad": entidad, "lotes": total_lotes, "archivos": len(rutas),
                   "filas_procesos": filas, "ids_proceso_distintos": procesos_distintos,
                   "filas_repetidas_por_contenido": filas - contenido_distinto,
                   "portafolios_encontrados": portafolios_encontrados.count(),
                   "portafolios_con_varios_procesos": multiples,
                   "contratos": total_contratos, "contratos_con_proceso": con_proceso,
                   "contratos_sin_proceso": total_contratos - con_proceso,
                   "cobertura_pct": round(100 * con_proceso / total_contratos, 2)}
        return resumen, detalles
    finally:
        contratos.unpersist()
        if procesos is not None:
            procesos.unpersist()

## 5. Ejecución y lectura de los resultados
Se revisan **IDARTES** e **IDIPRON** y se muestran dos tablas: resumen por entidad y detalle de páginas/registros por lote.
| Métrica auditada | IDARTES | IDIPRON |
| --- | ---: | ---: |
| Contratos | 3651 | 3233 |
| Lotes | 37 | 33 |
| Archivos de procesos | 62 | 44 |
| Filas de procesos | 3667 | 2888 |
| IDs de proceso distintos | 3629 | 2885 |
| Filas repetidas por contenido | 37 | 0 |
| Portafolios encontrados | 3588 | 2859 |
| Portafolios con varios IDs de proceso | 31 | 17 |
| Contratos con proceso | 3589 | 2860 |
| Contratos sin proceso | 62 | 373 |
| Cobertura | 98,30 % | 88,46 % |


In [0]:
resumenes, detalles = [], []
for entidad, nit, esperado in (("idartes", "900413030", 3651), ("idipron", "899999333", 3233)):
    resumen, detalle = revisar_entidad(entidad, nit, esperado)
    resumenes.append(resumen)
    detalles.extend(detalle)

In [0]:
display(spark.createDataFrame(resumenes).orderBy("entidad"))
display(spark.createDataFrame(detalles,
    "entidad string, lote int, paginas int, filas long, filas_ultima_pagina long").orderBy("entidad", "lote"))
print("Validacion de archivos, paginacion y pertenencia a lotes superada. No se modifico Raw.")
print("Los duplicados y contratos sin proceso se conservan para tratamiento posterior en Silver.")

archivos,cobertura_pct,contratos,contratos_con_proceso,contratos_sin_proceso,entidad,filas_procesos,filas_repetidas_por_contenido,ids_proceso_distintos,lotes,portafolios_con_varios_procesos,portafolios_encontrados
62,98.3,3651,3589,62,idartes,3667,37,3629,37,31,3588
44,88.46,3233,2860,373,idipron,2888,0,2885,33,17,2859


entidad,lote,paginas,filas,filas_ultima_pagina
idartes,1,1,90,90
idartes,2,1,87,87
idartes,3,1,92,92
idartes,4,1,97,97
idartes,5,2,100,0
idartes,6,2,100,0
idartes,7,2,101,1
idartes,8,2,101,1
idartes,9,2,100,0
idartes,10,2,100,0


Validacion de archivos, paginacion y pertenencia a lotes superada. No se modifico Raw.
Los duplicados y contratos sin proceso se conservan para tratamiento posterior en Silver.
